# Ensemble Learning - Boosting Companion Notebook

This notebook is the practical code companion for the boosting part of [`lecture_notes/12_ensemble_learning.pdf`](../../lecture_notes/12_ensemble_learning.pdf). The lecture notes explain the sequential bias-reduction logic; the cells below focus on how that logic shows up in real model behavior.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, log_loss
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42


This setup cell imports the ensemble models and evaluation tools used in the notebook. The experiments stay within `scikit-learn` so that the core boosting ideas can be explored without extra package dependencies.


## Build a Small Boosting Benchmark

We compare AdaBoost, Gradient Boosting, and Histogram-based Gradient Boosting on the same binary classification task.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)

ada = AdaBoostClassifier(
    estimator=stump,
    n_estimators=100,
    learning_rate=0.5,
    random_state=SEED,
)

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=SEED,
)

hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=100,
    early_stopping=False,
    random_state=SEED,
)

models = {
    "AdaBoost": ada,
    "Gradient Boosting": gb,
    "HistGradientBoosting": hgb,
}


This cell fixes the train/test split and defines three representative boosting models. AdaBoost uses decision stumps to make the sequential reweighting mechanism easy to interpret, Gradient Boosting represents the classical residual-fitting family, and Histogram-based Gradient Boosting is a more modern implementation designed for efficiency on larger tabular datasets.


In [ ]:
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_rows.append({
        "model": name,
        "cv_mean_accuracy": scores.mean(),
        "cv_std": scores.std(),
    })

pd.DataFrame(cv_rows).sort_values("cv_mean_accuracy", ascending=False).round(4)


Cross-validation gives the first model comparison under a shared evaluation procedure. Here the average accuracy is the headline number, but the fold-to-fold variation is also informative because boosting models can be sensitive to parameter choices and data perturbations.


In [ ]:
test_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    test_rows.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_log_loss": log_loss(y_test, y_prob),
    })

pd.DataFrame(test_rows).sort_values("test_accuracy", ascending=False).round(4)


This table adds held-out test accuracy and log loss. Including log loss is useful because boosting often changes not only class labels but also the confidence structure of the predicted probabilities.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


The confusion matrices are the easiest way to see whether the boosting variants are making the same kinds of mistakes. Even when accuracies are close, the error profile may differ in a way that matters for downstream use.


## Inspect Boosting Round by Round

One practical advantage of gradient boosting is that it exposes staged predictions, making the sequential construction of the ensemble visible.


In [ ]:
staged_log_loss = []
staged_accuracy = []

for y_prob in gb.staged_predict_proba(X_test):
    staged_log_loss.append(log_loss(y_test, y_prob))
    staged_accuracy.append(accuracy_score(y_test, np.argmax(y_prob, axis=1)))

staged_df = pd.DataFrame({
    "round": np.arange(1, len(staged_log_loss) + 1),
    "log_loss": staged_log_loss,
    "accuracy": staged_accuracy,
})
staged_df.head()


This cell records the test-set behavior of the gradient boosting model after each boosting round. The resulting table is mainly a bridge to the plots below, where the sequential nature of boosting becomes visible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(staged_df["round"], staged_df["log_loss"], marker="o", markersize=3)
axes[0].set_xlabel("boosting round")
axes[0].set_ylabel("test log loss")
axes[0].set_title("Log loss across boosting rounds")

axes[1].plot(staged_df["round"], staged_df["accuracy"], marker="o", markersize=3)
axes[1].set_xlabel("boosting round")
axes[1].set_ylabel("test accuracy")
axes[1].set_title("Accuracy across boosting rounds")

plt.tight_layout()
plt.show()


These curves are useful because they show that boosting is not just a black-box final model. The ensemble is built incrementally, and performance often improves for a while before stabilizing. On harder problems, this kind of plot is also a natural starting point for early-stopping decisions.


## Learning Rate as a Practical Hyperparameter

The notes discuss the role of the learning rate conceptually. Here we inspect it empirically while keeping the number of estimators fixed.


In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.3]
lr_rows = []

for lr in learning_rates:
    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=2,
        random_state=SEED,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    lr_rows.append({
        "learning_rate": lr,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_log_loss": log_loss(y_test, y_prob),
    })

lr_df = pd.DataFrame(lr_rows)
lr_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(lr_df["learning_rate"], lr_df["test_accuracy"], marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("learning rate")
axes[0].set_ylabel("test accuracy")
axes[0].set_title("Accuracy vs learning rate")

axes[1].plot(lr_df["learning_rate"], lr_df["test_log_loss"], marker="o")
axes[1].set_xscale("log")
axes[1].set_xlabel("learning rate")
axes[1].set_ylabel("test log loss")
axes[1].set_title("Log loss vs learning rate")

plt.tight_layout()
plt.show()


This experiment makes the learning-rate trade-off concrete. Smaller values usually make each stage more conservative, which can improve generalization if enough rounds are available; larger values make the ensemble move faster but can become harder to control.


## Histogram-Based Gradient Boosting and Early Stopping

The old notebook also included `HistGradientBoostingClassifier`. It is worth keeping because it shows a modern boosting implementation with built-in support for early stopping.


In [ ]:
hgb_no_es = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=200,
    early_stopping=False,
    random_state=SEED,
)

hgb_es = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=SEED,
)

hgb_no_es.fit(X_train, y_train)
hgb_es.fit(X_train, y_train)

pd.DataFrame([
    {
        "model": "HistGB without early stopping",
        "effective_iterations": hgb_no_es.n_iter_,
        "test_accuracy": accuracy_score(y_test, hgb_no_es.predict(X_test)),
    },
    {
        "model": "HistGB with early stopping",
        "effective_iterations": hgb_es.n_iter_,
        "test_accuracy": accuracy_score(y_test, hgb_es.predict(X_test)),
    },
]).round(4)


This comparison shows whether early stopping actually truncates the boosting process on this dataset. The key number is `effective_iterations`: if it is much smaller with `early_stopping=True`, the model found that continuing the sequence was no longer improving validation performance enough to justify more rounds.
